# Phase 07A - AUGR-VQA Paper Naming Harmonization Final Release

This notebook is the last-run paper-release notebook for the AUGR-VQA project.

It does not train, evaluate, recalibrate, or recalculate the research results. It reads completed Google Drive artifacts, validates the source paths, copies paper-facing artifacts into a new release folder, and harmonizes copied filenames and display labels with the manuscript naming convention:

```text
AUGR-VQA = QAdp-DG-PRUGTM + Q-CUR
```

Preservation policy:

- Previous notebooks are not edited.
- Previous result folders are not edited.
- Model checkpoints, raw predictions, and source artifacts remain untouched.
- Old implementation names are preserved only as provenance/source-path evidence in the copied release manifest or provenance fields.


## Dual Environment Compatibility Setup & Path Configuration


In [ ]:
# ── DUAL ENVIRONMENT COMPATIBILITY & DEPENDENCY SETUP ────────────────────────
import os
import sys
from pathlib import Path

def resolve_project_environment(mount_point: str = "/content/drive") -> tuple[Path, Path]:
    try:
        import google.colab
        from google.colab import drive
        drive.mount(mount_point)
        project_root = Path(mount_point) / "MyDrive" / "AUGR-VQA"
        temp_dir = Path("/content")
        print("Running in Google Colab environment.")
    except ImportError:
        # Running locally (parent of notebooks directory)
        project_root = Path(os.getcwd()).parent.resolve()
        temp_dir = project_root / "temp"
        temp_dir.mkdir(parents=True, exist_ok=True)
        print("Running in Local environment.")
    return project_root, temp_dir

PROJECT_ROOT, TEMP_DIR = resolve_project_environment()
# ─────────────────────────────────────────────────────────────────────────────

from __future__ import annotations

import csv
import hashlib
import json
import os
import re
import shutil
import time
from datetime import datetime
from pathlib import Path
from typing import Any

import matplotlib.pyplot as plt
import pandas as pd


def running_in_colab() -> bool:
    try:
        import google.colab
        return True
    except Exception:
        return False


if running_in_colab():
    from google.colab import drive
#     drive.mount("/content/drive")


# PROJECT_ROOT definition overridden by helper

SOURCE_ROOTS = {
    "phase03c_audit": Path("phase_3/p3c_dataset_benchmark_readiness_audit/btumqa_225k_clean_metadata_bias_audit"),
    "phase03d_baseline": Path("phase_3/p3d_qa_shortcut_bias_diagnostic_baselines/btumqa_225k_clean_metadata_shortcut_audit/qadp_prugtm_baseline"),
    "phase03g_input_ablation": Path("phase_3/p3d_qa_shortcut_bias_diagnostic_baselines/btumqa_225k_clean_metadata_shortcut_audit/qadp_prugtm_input_ablation"),
    "phase05b_final_eval": Path("phase_5/p5b_final_evaluation_ablation_calibration/btumqa_225k_clean_metadata_four_seeds_final_comparison"),
    "phase05c_boot2000": Path("phase_5/p5c_statistical_confidence_slice_robustness/btumqa_225k_clean_metadata_four_seeds_statistical_robustness_boot2000"),
    "phase05d_modern_baselines": Path("phase_5/p5d_modern_baseline_comparison_models"),
    "phase06a_unified": Path("phase_6/p6a_models_unified_comparison_results_showcase/btumqa_225k_models_unified_comparison_results_showcase"),
}

RELEASE_BASE_DIR = PROJECT_ROOT / "phase_7" / "p7a_paper_ready_augr_vqa_release" / "btumqa_225k_clean_metadata_four_seeds"
OUTPUT_ROOT = RELEASE_BASE_DIR
ARCHITECTURE_FIGURE_SOURCE_DIR = None  # Optional: set to a Drive/Colab folder if architecture figures are uploaded for this release.
RUN_ID = datetime.now().strftime("%Y%m%d_%H%M%S")

# Keep the public release folder stable, but archive an existing release if present.
ARCHIVE_EXISTING_OUTPUT = True
REQUIRE_ALL_RESULT_FIGURES_REGENERATED = True

EXPECTED_MIN_COUNTS = {
    "phase03c_audit": {".csv": 10, ".json": 3},
    "phase03d_baseline": {".csv": 9, ".png": 5, ".json": 3},
    "phase03g_input_ablation": {".csv": 22, ".png": 4, ".json": 3},
    "phase05b_final_eval": {".csv": 25, ".jsonl": 18, ".png": 14, ".json": 5},
    "phase05c_boot2000": {".csv": 5, ".png": 8, ".json": 3},
    "phase06a_unified": {".csv": 7, ".png": 12, ".json": 2},
}

print(f"Running in Colab: {running_in_colab()}")
print(f"PROJECT_ROOT: {PROJECT_ROOT}")
print(f"RELEASE_BASE_DIR: {RELEASE_BASE_DIR}")
print(f"OUTPUT_ROOT: {OUTPUT_ROOT}")
print(f"ARCHITECTURE_FIGURE_SOURCE_DIR: {ARCHITECTURE_FIGURE_SOURCE_DIR}")
print(f"REQUIRE_ALL_RESULT_FIGURES_REGENERATED: {REQUIRE_ALL_RESULT_FIGURES_REGENERATED}")


Mounted at /content/drive
Running in Colab: True
PROJECT_ROOT: /content/drive/MyDrive/AMIR Lab/Research Assistant Intern/research_01_brain_tumor_VQA_medical_imaging
RELEASE_BASE_DIR: /content/drive/MyDrive/AMIR Lab/Research Assistant Intern/research_01_brain_tumor_VQA_medical_imaging/phase_7_paper_ready_augr_vqa_release/btumqa_225k_clean_metadata_four_seeds
OUTPUT_ROOT: /content/drive/MyDrive/AMIR Lab/Research Assistant Intern/research_01_brain_tumor_VQA_medical_imaging/phase_7_paper_ready_augr_vqa_release/btumqa_225k_clean_metadata_four_seeds/results
ARCHITECTURE_FIGURE_SOURCE_DIR: None
REQUIRE_ALL_RESULT_FIGURES_REGENERATED: True


## Validate Source Paths And Artifact Counts


In [ ]:
def count_files(path: Path) -> int:
    if path.is_file():
        return 1
    if path.is_dir():
        return sum(1 for item in path.rglob("*") if item.is_file())
    return 0


def suffix_counts(path: Path) -> dict[str, int]:
    counts = {}
    if path.is_file():
        counts[path.suffix.lower() or "<none>"] = 1
        return counts
    if path.is_dir():
        for item in path.rglob("*"):
            if item.is_file():
                suffix = item.suffix.lower() or "<none>"
                counts[suffix] = counts.get(suffix, 0) + 1
    return counts


validation_rows = []
missing_required = []

if not PROJECT_ROOT.exists():
    raise FileNotFoundError(
        f"PROJECT_ROOT does not exist: {PROJECT_ROOT}\n"
        "Update PROJECT_ROOT to your Google Drive project folder before continuing."
    )

for key, rel_path in SOURCE_ROOTS.items():
    full_path = PROJECT_ROOT / rel_path
    exists = full_path.exists()
    file_count = count_files(full_path) if exists else 0
    counts = suffix_counts(full_path) if exists else {}
    expected = EXPECTED_MIN_COUNTS.get(key, {})
    count_failures = []
    for suffix, expected_count in expected.items():
        observed = counts.get(suffix, 0)
        if observed < expected_count:
            count_failures.append(f"{suffix}: observed {observed}, expected >= {expected_count}")
    validation_rows.append(
        {
            "source_key": key,
            "relative_path": str(rel_path),
            "full_path": str(full_path),
            "exists": exists,
            "file_count": file_count,
            "suffix_counts": json.dumps(counts, sort_keys=True),
            "expected_min_counts": json.dumps(expected, sort_keys=True),
            "count_validation_passed": not count_failures,
            "count_failures": "; ".join(count_failures),
        }
    )
    if not exists:
        missing_required.append(key)
    if count_failures:
        missing_required.append(f"{key} count validation")

validation_df = pd.DataFrame(validation_rows)
display(validation_df)

if missing_required:
    raise FileNotFoundError(f"Source validation failed: {missing_required}")

print("All required source paths and expected artifact counts passed. Review the checklist above before running the copy cells.")


,source_key,relative_path,full_path,exists,file_count,suffix_counts,expected_min_counts,count_validation_passed,count_failures
0,phase03c_audit,phase_3c_dataset_benchmark_readiness_audit/btu...,/content/drive/MyDrive/AMIR Lab/Research Assis...,True,18,"{"".csv"": 10, "".json"": 3, "".png"": 5}","{"".csv"": 10, "".json"": 3}",True,
1,phase03d_baseline,phase_3d_qa_shortcut_bias_diagnostic_baselines...,/content/drive/MyDrive/AMIR Lab/Research Assis...,True,18,"{"".csv"": 9, "".json"": 3, "".pkl"": 1, "".png"": 5}","{"".csv"": 9, "".json"": 3, "".png"": 5}",True,
2,phase03g_input_ablation,phase_3d_qa_shortcut_bias_diagnostic_baselines...,/content/drive/MyDrive/AMIR Lab/Research Assis...,True,43,"{"".csv"": 22, "".json"": 3, "".png"": 4, "".pt"": 14}","{"".csv"": 22, "".json"": 3, "".png"": 4}",True,
3,phase05b_final_eval,phase_5b_final_evaluation_ablation_calibration...,/content/drive/MyDrive/AMIR Lab/Research Assis...,True,110,"{"".csv"": 57, "".joblib"": 16, "".json"": 5, "".json...","{"".csv"": 25, "".json"": 5, "".jsonl"": 18, "".png"":...",True,
4,phase05c_boot2000,phase_5c_statistical_confidence_slice_robustne...,/content/drive/MyDrive/AMIR Lab/Research Assis...,True,16,"{"".csv"": 5, "".json"": 3, "".png"": 8}","{"".csv"": 5, "".json"": 3, "".png"": 8}",True,
5,phase05d_modern_baselines,phase_5d_modern_baseline_comparison_models,/content/drive/MyDrive/AMIR Lab/Research Assis...,True,10791,"{"".csv"": 373, "".json"": 162, "".png"": 22, "".pt"":...",{},True,
6,phase06a_unified,phase_06a_models_unified_comparison_results_sh...,/content/drive/MyDrive/AMIR Lab/Research Assis...,True,21,"{"".csv"": 7, "".json"": 2, "".png"": 12}","{"".csv"": 7, "".json"": 2, "".png"": 12}",True,


All required source paths and expected artifact counts passed. Review the checklist above before running the copy cells.


## Canonical Naming Map


In [ ]:
FRAMEWORK_STATEMENT = "AUGR-VQA = QAdp-DG-PRUGTM + Q-CUR"

DISPLAY_REPLACEMENTS = [
    ("QAdp-PRUGTM-Hybrid + Q-CUR", "AUGR-VQA"),
    ("QAdp-PRUGTM-Hybrid", "QAdp-DG-PRUGTM"),
    ("QAdp PRUGTM Hybrid + Q-CUR", "AUGR-VQA"),
    ("qadp_prugtm_hybrid_qcur", "augr_vqa"),
    ("qadp_prugtm_hybrid_clean", "qadp_dg_prugtm"),
    ("qadp_prugtm_hybrid", "qadp_dg_prugtm"),
    ("QAdp PRUGTM Hybrid", "QAdp-DG-PRUGTM"),
    ("Adp-PRUGTM-Hybrid", "QAdp-DG-PRUGTM"),
    ("DenseNet121 MLP", "DenseNet121"),
    ("ResNet50 MLP", "ResNet50"),
    ("EfficientNet-B0 MLP", "EfficientNet-B0"),
    ("ViT-B/16 MLP", "ViT-B/16"),
    ("Swin-T MLP", "Swin-T"),
    ("Native frozen BiomedCLIP", "BiomedCLIP-A3"),
    ("Supervised frozen BiomedCLIP", "BiomedCLIP-A4"),
    ("BiomedCLIP dual-view naturalized Global-23", "BiomedCLIP-A3"),
    ("BiomedCLIP frozen-feature supervised", "BiomedCLIP-A4"),
    ("BiomedCLIP Frozen Question+Image MLP", "BiomedCLIP-A4"),
    ("BiomedCLIP Frozen Image-Only MLP", "BiomedCLIP-A4 Image-only"),
    ("BiomedCLIP Naturalized Global-23 Ensemble", "BiomedCLIP-A3"),
    ("DenseNet121 Question+Image MLP", "DenseNet121"),
    ("ResNet50 Question+Image MLP", "ResNet50"),
    ("EfficientNet-B0 Question+Image MLP", "EfficientNet-B0"),
    ("ViT-B/16 Question+Image MLP", "ViT-B/16"),
    ("Swin-T Question+Image MLP", "Swin-T"),
    ("DenseNet121 Image-Only MLP", "DenseNet121 Image-only"),
    ("ResNet50 Image-Only MLP", "ResNet50 Image-only"),
    ("EfficientNet-B0 Image-Only MLP", "EfficientNet-B0 Image-only"),
    ("ViT-B/16 Image-Only MLP", "ViT-B/16 Image-only"),
    ("Swin-T Image-Only MLP", "Swin-T Image-only"),
]

FILENAME_REPLACEMENTS = [
    ("qadp_prugtm_hybrid_qcur", "augr_vqa"),
    ("qadp_prugtm_hybrid_clean", "qadp_dg_prugtm"),
    ("qadp_prugtm_hybrid", "qadp_dg_prugtm"),
    ("qadp_prugtm", "augr_vqa"),
    ("shortcut_vs_qadp", "shortcut_vs_augr_vqa"),
    ("phase03d_clean_qadp", "phase03d_clean_augr_vqa"),
    ("phase03g_qadp", "phase03g_augr_vqa"),
    ("qcur", "qcur"),
]

PROVENANCE_HINTS = (
    "source",
    "path",
    "file",
    "dir",
    "folder",
    "parent",
    "artifact",
    "report",
    "metric",
    "manifest",
    "provenance",
)

REQUIRED_TABLE_SCHEMAS = {
    "phase_3/p3d_qa_shortcut_bias_diagnostic_baselines/btumqa_225k_clean_metadata_shortcut_audit/qadp_prugtm_baseline/tables/phase03d_clean_qadp_confidence_by_correctness.csv": [
        "qadp_correct", "rows", "raw_confidence_mean", "qcur_confidence_mean"
    ],
    "phase_3/p3d_qa_shortcut_bias_diagnostic_baselines/btumqa_225k_clean_metadata_shortcut_audit/qadp_prugtm_baseline/tables/phase03d_clean_qadp_error_concentration.csv": [
        "slice_column", "slice_value", "rows", "qadp_error_rate", "mean_qcur_confidence"
    ],
    "phase_3/p3d_qa_shortcut_bias_diagnostic_baselines/btumqa_225k_clean_metadata_shortcut_audit/qadp_prugtm_baseline/tables/phase03d_clean_qadp_shortcut_gap_by_slice.csv": [
        "model_label", "slice_column", "slice_value", "rows", "macro_f1", "qadp_minus_best_clean_safe_macro_f1"
    ],
    "phase_3/p3d_qa_shortcut_bias_diagnostic_baselines/btumqa_225k_clean_metadata_shortcut_audit/qadp_prugtm_baseline/tables/phase03d_clean_review_flag_usefulness.csv": [
        "review_flag", "rows", "row_share", "qadp_error_rate", "mean_qcur_confidence"
    ],
    "phase_3/p3d_qa_shortcut_bias_diagnostic_baselines/btumqa_225k_clean_metadata_shortcut_audit/qadp_prugtm_baseline/tables/phase03d_clean_shortcut_vs_qadp_overall_metrics.csv": [
        "model_key", "model_label", "accuracy", "macro_f1", "weighted_f1"
    ],
    "phase_3/p3d_qa_shortcut_bias_diagnostic_baselines/btumqa_225k_clean_metadata_shortcut_audit/qadp_prugtm_input_ablation/tables/phase03g_clean_input_ablation_aggregated_metrics.csv": [
        "model_key", "model_label", "test_accuracy_mean", "test_macro_f1_mean", "test_weighted_f1_mean"
    ],
    "phase_3/p3d_qa_shortcut_bias_diagnostic_baselines/btumqa_225k_clean_metadata_shortcut_audit/qadp_prugtm_input_ablation/tables/phase03g_clean_input_ablation_qadp_gap_metrics.csv": [
        "metric", "reference_model_label", "qadp_value", "reference_value", "qadp_minus_reference"
    ],
    "phase_3/p3d_qa_shortcut_bias_diagnostic_baselines/btumqa_225k_clean_metadata_shortcut_audit/qadp_prugtm_input_ablation/tables/phase03g_clean_input_ablation_slice_aggregated_metrics.csv": [
        "model_key", "model_label", "slice_column", "slice_value", "macro_f1_mean"
    ],
    "phase_5/p5b_final_evaluation_ablation_calibration/btumqa_225k_clean_metadata_four_seeds_final_comparison/tables/phase5b_clean_metadata_four_seeds_answer_backbone_table.csv": [
        "backbone_key", "backbone", "test_accuracy_mean", "test_macro_f1_mean", "test_weighted_f1_mean"
    ],
    "phase_5/p5b_final_evaluation_ablation_calibration/btumqa_225k_clean_metadata_four_seeds_final_comparison/tables/phase5b_clean_metadata_four_seeds_ablation_comparison_table.csv": [
        "backbone_key", "backbone", "test_accuracy_mean", "test_macro_f1_mean", "test_weighted_f1_mean"
    ],
    "phase_5/p5b_final_evaluation_ablation_calibration/btumqa_225k_clean_metadata_four_seeds_final_comparison/tables/phase5b_clean_metadata_four_seeds_calibration_aggregated.csv": [
        "backbone_key", "backbone", "method", "ece__mean", "brier__mean", "nll__mean", "aurc__mean"
    ],
    "phase_5/p5b_final_evaluation_ablation_calibration/btumqa_225k_clean_metadata_four_seeds_final_comparison/tables/phase5b_clean_metadata_four_seeds_key_slice_calibration.csv": [
        "backbone_key", "backbone", "method", "slice_type", "slice_value", "ece__mean", "brier__mean", "nll__mean", "aurc__mean"
    ],
    "phase_5/p5c_statistical_confidence_slice_robustness/btumqa_225k_clean_metadata_four_seeds_statistical_robustness_boot2000/tables/phase5c_clean_metadata_four_seeds_bootstrap_ci_global_metrics.csv": [
        "backbone_key", "backbone", "method", "metric", "observed_mean", "ci_lower", "ci_upper"
    ],
    "phase_5/p5c_statistical_confidence_slice_robustness/btumqa_225k_clean_metadata_four_seeds_statistical_robustness_boot2000/tables/phase5c_clean_metadata_four_seeds_bootstrap_pairwise_differences.csv": [
        "comparison_id", "metric", "left_backbone", "right_backbone", "observed_difference", "ci_lower", "ci_upper"
    ],
    "phase_5/p5c_statistical_confidence_slice_robustness/btumqa_225k_clean_metadata_four_seeds_statistical_robustness_boot2000/tables/phase5c_clean_metadata_four_seeds_key_slice_answer_comparison.csv": [
        "slice_type", "slice_value", "backbone_key", "backbone", "macro_f1_mean"
    ],
    "phase_5/p5c_statistical_confidence_slice_robustness/btumqa_225k_clean_metadata_four_seeds_statistical_robustness_boot2000/tables/phase5c_clean_metadata_four_seeds_key_slice_calibration_comparison.csv": [
        "slice_type", "slice_value", "backbone_key", "backbone", "method", "ece_mean", "brier_mean", "nll_mean", "aurc_mean"
    ],
    "phase_5/p5c_statistical_confidence_slice_robustness/btumqa_225k_clean_metadata_four_seeds_statistical_robustness_boot2000/tables/phase5c_clean_metadata_four_seeds_key_slice_qcur_gain_summary.csv": [
        "slice_type", "slice_value", "backbone_key", "backbone", "ece_gain", "brier_gain", "nll_gain", "aurc_gain"
    ],
    "phase_6/p6a_models_unified_comparison_results_showcase/btumqa_225k_models_unified_comparison_results_showcase/tables/unified_main_paper_model_comparison.csv": [
        "model_label", "accuracy", "macro_f1", "weighted_f1", "ece", "brier", "nll", "aurc"
    ],
    "phase_6/p6a_models_unified_comparison_results_showcase/btumqa_225k_models_unified_comparison_results_showcase/tables/unified_ablation_comparison.csv": [
        "Variant", "Accuracy", "Macro-F1", "Weighted-F1", "Delta Macro-F1 vs Proposed"
    ],
    "phase_6/p6a_models_unified_comparison_results_showcase/btumqa_225k_models_unified_comparison_results_showcase/tables/unified_calibration_reliability_comparison.csv": [
        "model_label", "ece", "brier", "nll", "aurc", "mean_confidence"
    ],
    "phase_6/p6a_models_unified_comparison_results_showcase/btumqa_225k_models_unified_comparison_results_showcase/tables/unified_statistical_robustness_comparison.csv": [
        "backbone", "method", "metric", "observed_mean", "ci_lower", "ci_upper"
    ],
}


def apply_display_replacements(text: str) -> tuple[str, int]:
    total = 0
    new_text = text
    for old, new in DISPLAY_REPLACEMENTS:
        hits = new_text.count(old)
        if hits:
            new_text = new_text.replace(old, new)
            total += hits
    malformed = "AUGR-VQA = QAdp-DG-PRUGTMCNN"
    if malformed in new_text:
        new_text = new_text.replace(malformed, "AUGR-VQA = QAdp-DG-PRUGTM + Q-CUR. CNN")
        total += 1
    malformed_qcur = "AUGR-VQA = QAdp-DG-PRUGTM + Q-CURCNN"
    if malformed_qcur in new_text:
        new_text = new_text.replace(malformed_qcur, "AUGR-VQA = QAdp-DG-PRUGTM + Q-CUR. CNN")
        total += 1
    malformed_old = "AUGR-VQA = Adp-PRUGTM-Hybrid + Q-CURCNN"
    if malformed_old in new_text:
        new_text = new_text.replace(malformed_old, "AUGR-VQA = QAdp-DG-PRUGTM + Q-CUR. CNN")
        total += 1
    return new_text, total


def paper_ready_filename(name: str) -> str:
    new_name = name
    for old, new in FILENAME_REPLACEMENTS:
        new_name = new_name.replace(old, new)
    new_name = new_name.replace("phase5b_clean_metadata_four_seeds", "phase5b_augr_vqa")
    new_name = new_name.replace("phase5c_clean_metadata_four_seeds", "phase5c_augr_vqa")
    new_name = new_name.replace("phase06a_models_unified_results", "phase06a_augr_vqa_models_unified_results")
    new_name = new_name.replace("unified_", "phase06a_augr_vqa_unified_")
    new_name = new_name.replace("selected_model", "augr_vqa_selected_model")
    return new_name


def is_provenance_key(key: str) -> bool:
    lower = key.lower()
    return any(hint in lower for hint in PROVENANCE_HINTS)


print(FRAMEWORK_STATEMENT)
print("Canonical naming map loaded.")


AUGR-VQA = QAdp-DG-PRUGTM + Q-CUR
Canonical naming map loaded.


## Build Paper-Ready Copy Plan


In [ ]:
table_schema_rows = []
schema_failures = []
for rel_csv, required_columns in REQUIRED_TABLE_SCHEMAS.items():
    path = PROJECT_ROOT / rel_csv
    exists = path.exists()
    observed_columns = []
    missing_columns = list(required_columns)
    if exists:
        try:
            observed_columns = list(pd.read_csv(path, nrows=0).columns)
            missing_columns = [col for col in required_columns if col not in observed_columns]
        except Exception as exc:
            missing_columns = [f"READ_ERROR: {type(exc).__name__}: {exc}"]
    table_schema_rows.append(
        {
            "relative_csv": rel_csv,
            "exists": exists,
            "required_columns": json.dumps(required_columns),
            "observed_columns": json.dumps(observed_columns),
            "missing_columns": json.dumps(missing_columns),
            "schema_passed": exists and not missing_columns,
        }
    )
    if (not exists) or missing_columns:
        schema_failures.append(rel_csv)

table_schema_validation_df = pd.DataFrame(table_schema_rows)
display(table_schema_validation_df)
if schema_failures:
    raise ValueError(f"Required table schema validation failed for: {schema_failures}")

CATEGORY_DIRS = {
    "tables": "tables",
    "figures": "figures",
    "reports": "reports",
    "evidence": "evidence",
    "manifests": "manifests",
    "architecture_figures": "figures/architecture",
    "source_figure_copies": "figures/source_copies",
    "reference_figure_copies": "figures/reference_copies",
    "regenerated_figures": "figures/regenerated_paper_ready",
}

SOURCE_CATEGORY_RULES = [
    ("phase03c_audit", "tables"),
    ("phase03d_baseline", "tables"),
    ("phase03g_input_ablation", "tables"),
    ("phase05b_final_eval", "tables"),
    ("phase05c_boot2000", "tables"),
    ("phase05d_modern_baselines", "tables"),
    ("phase06a_unified", "tables"),
]

ALLOWED_SUBFOLDERS = {"tables", "figures", "reports", "explanation_ready", "predictions", "results"}
ALLOWED_SUFFIXES = {".csv", ".json", ".jsonl", ".md", ".txt", ".png", ".jpg", ".jpeg", ".svg", ".pdf", ".fig"}
STRICT_REGENERATED_FIGURE_SOURCE_KEYS = {
    "phase03d_baseline",
    "phase03g_input_ablation",
    "phase05b_final_eval",
    "phase05c_boot2000",
    "phase06a_unified",
}


def clean_parent_path(category: str, source_key: str, parent: Path) -> Path:
    # 1. Strip redundant category names from the parent directories
    parts = list(parent.parts)
    cleaned_parts = []
    for part in parts:
        if part in {category, "tables", "reports", "figures", "predictions", "explanation_ready"}:
            continue
        cleaned_parts.append(part)
        
    # 2. Map long modern baseline subdirectories to shortened forms
    if source_key == "phase05d_modern_baselines":
        mapped_parts = []
        for part in cleaned_parts:
            if part == "cnn_model_comparison":
                mapped_parts.append("cnn")
            elif part == "attention_model_comparison":
                mapped_parts.append("attention")
            elif part == "vlm_comparison":
                mapped_parts.append("vlm")
            elif part == "phase05d_b1_cnn_frozen_feature_supervised_baselines":
                mapped_parts.append("b1_cnn")
            elif part == "phase05d_a3_biomedclip_dual_view_naturalized_global23_ensemble":
                mapped_parts.append("a3_biomed")
            elif part == "phase05d_a4_biomedclip_frozen_feature_supervised_baseline":
                mapped_parts.append("a4_biomed")
            elif part == "phase05d_c1_attention_frozen_feature_supervised_baselines":
                mapped_parts.append("c1_attention")
            else:
                mapped_parts.append(part)
        cleaned_parts = mapped_parts
        
    return Path(*cleaned_parts) if cleaned_parts else Path(".")


def classify_artifact(source_key: str, path: Path) -> str | None:
    parts = set(path.relative_to(PROJECT_ROOT / SOURCE_ROOTS[source_key]).parts)
    if "explanation_ready" in parts:
        return "evidence"
    if "tables" in parts:
        return "tables"
    if "figures" in parts:
        if source_key not in STRICT_REGENERATED_FIGURE_SOURCE_KEYS:
            return "reference_figure_copies"
        return "source_figure_copies"
    if "reports" in parts:
        return "reports"
    if source_key == "phase05d_modern_baselines" and path.suffix.lower() in {".csv", ".json", ".jsonl", ".md", ".txt", ".png", ".jpg", ".jpeg", ".svg", ".pdf"}:
        if "predictions" in parts or "results" in parts:
            return "evidence"
        if path.suffix.lower() in {".png", ".jpg", ".jpeg", ".svg"}:
            return "reference_figure_copies"
        return "reports" if path.suffix.lower() in {".json", ".md", ".txt", ".pdf"} else "tables"
    return None


copy_plan = []
for source_key, rel_root in SOURCE_ROOTS.items():
    source_root = PROJECT_ROOT / rel_root
    files = [source_root] if source_root.is_file() else sorted(item for item in source_root.rglob("*") if item.is_file())
    for src in files:
        if src.suffix.lower() not in ALLOWED_SUFFIXES:
            continue
        category = classify_artifact(source_key, src)
        if category is None:
            continue
        rel_to_source = src.relative_to(source_root)
        paper_name = paper_ready_filename(src.name)
        
        if category == "source_figure_copies":
            cleaned_parent = clean_parent_path("source_figure_copies", source_key, rel_to_source.parent)
            dst_rel = Path("figures") / "source_copies" / source_key / cleaned_parent / paper_name
        elif category == "reference_figure_copies":
            cleaned_parent = clean_parent_path("reference_figure_copies", source_key, rel_to_source.parent)
            dst_rel = Path("figures") / "reference_copies" / source_key / cleaned_parent / paper_name
        else:
            # Preserve phase/source grouping enough to avoid collisions while still normalizing filenames.
            cleaned_parent = clean_parent_path(category, source_key, rel_to_source.parent)
            dst_rel = Path(CATEGORY_DIRS[category]) / source_key / cleaned_parent / paper_name
            
        copy_plan.append(
            {
                "source_key": source_key,
                "category": category,
                "source_path": src,
                "destination_relative": dst_rel,
                "destination_path": OUTPUT_ROOT / dst_rel,
                "source_size_bytes": src.stat().st_size,
            }
        )

architecture_source_dir = Path(ARCHITECTURE_FIGURE_SOURCE_DIR) if ARCHITECTURE_FIGURE_SOURCE_DIR else None
architecture_context_rows = []
if architecture_source_dir is not None:
    if not architecture_source_dir.exists():
        raise FileNotFoundError(f"ARCHITECTURE_FIGURE_SOURCE_DIR does not exist: {architecture_source_dir}")
    for src in sorted(item for item in architecture_source_dir.rglob("*") if item.is_file()):
        if src.suffix.lower() not in {".png", ".svg", ".fig"}:
            continue
        paper_name = paper_ready_filename(src.name)
        dst_rel = Path("figures") / "architecture" / paper_name
        copy_plan.append(
            {
                "source_key": "architecture_optional",
                "category": "architecture_figures",
                "source_path": src,
                "destination_relative": dst_rel,
                "destination_path": OUTPUT_ROOT / dst_rel,
                "source_size_bytes": src.stat().st_size,
            }
        )
        architecture_context_rows.append({"architecture_source_path": str(src), "status": "planned_copy"})
else:
    architecture_context_rows.append(
        {
            "architecture_source_path": "",
            "status": "not_provided_in_colab",
            "notes": "Set ARCHITECTURE_FIGURE_SOURCE_DIR to copy local/uploaded method figures into figures/architecture.",
        }
    )

architecture_context_df = pd.DataFrame(architecture_context_rows)
display(architecture_context_df)

copy_plan_df = pd.DataFrame(
    [{**row, "source_path": str(row["source_path"]), "destination_path": str(row["destination_path"])} for row in copy_plan]
)
display(copy_plan_df.head(25))
print(f"Planned artifacts: {len(copy_plan_df)}")
print(copy_plan_df["category"].value_counts().to_string())


,relative_csv,exists,required_columns,observed_columns,missing_columns,schema_passed
0,phase_3d_qa_shortcut_bias_diagnostic_baselines...,True,"[""qadp_correct"", ""rows"", ""raw_confidence_mean""...","[""qadp_correct"", ""rows"", ""raw_confidence_mean""...",[],True
1,phase_3d_qa_shortcut_bias_diagnostic_baselines...,True,"[""slice_column"", ""slice_value"", ""rows"", ""qadp_...","[""slice_column"", ""slice_value"", ""rows"", ""qadp_...",[],True
2,phase_3d_qa_shortcut_bias_diagnostic_baselines...,True,"[""model_label"", ""slice_column"", ""slice_value"",...","[""model_key"", ""model_label"", ""source"", ""policy...",[],True
3,phase_3d_qa_shortcut_bias_diagnostic_baselines...,True,"[""review_flag"", ""rows"", ""row_share"", ""qadp_err...","[""review_flag"", ""rows"", ""row_share"", ""qadp_err...",[],True
4,phase_3d_qa_shortcut_bias_diagnostic_baselines...,True,"[""model_key"", ""model_label"", ""accuracy"", ""macr...","[""model_key"", ""model_label"", ""split"", ""source""...",[],True
5,phase_3d_qa_shortcut_bias_diagnostic_baselines...,True,"[""model_key"", ""model_label"", ""test_accuracy_me...","[""model_key"", ""model_label"", ""num_runs"", ""use_...",[],True
6,phase_3d_qa_shortcut_bias_diagnostic_baselines...,True,"[""metric"", ""reference_model_label"", ""qadp_valu...","[""metric"", ""reference_model_key"", ""reference_m...",[],True
7,phase_3d_qa_shortcut_bias_diagnostic_baselines...,True,"[""model_key"", ""model_label"", ""slice_column"", ""...","[""model_key"", ""model_label"", ""slice_column"", ""...",[],True
8,phase_5b_final_evaluation_ablation_calibration...,True,"[""backbone_key"", ""backbone"", ""test_accuracy_me...","[""rank"", ""backbone_key"", ""backbone"", ""num_runs...",[],True
9,phase_5b_final_evaluation_ablation_calibration...,True,"[""backbone_key"", ""backbone"", ""test_accuracy_me...","[""rank"", ""backbone_key"", ""backbone"", ""num_runs...",[],True


,architecture_source_path,status,notes
0,,not_provided_in_colab,Set ARCHITECTURE_FIGURE_SOURCE_DIR to copy loc...


,source_key,category,source_path,destination_relative,destination_path,source_size_bytes
0,phase03c_audit,reference_figure_copies,/content/drive/MyDrive/AMIR Lab/Research Assis...,figures/reference_copies/phase03c_audit/figure...,/content/drive/MyDrive/AMIR Lab/Research Assis...,158839
1,phase03c_audit,reference_figure_copies,/content/drive/MyDrive/AMIR Lab/Research Assis...,figures/reference_copies/phase03c_audit/figure...,/content/drive/MyDrive/AMIR Lab/Research Assis...,68753
2,phase03c_audit,reference_figure_copies,/content/drive/MyDrive/AMIR Lab/Research Assis...,figures/reference_copies/phase03c_audit/figure...,/content/drive/MyDrive/AMIR Lab/Research Assis...,233766
3,phase03c_audit,reference_figure_copies,/content/drive/MyDrive/AMIR Lab/Research Assis...,figures/reference_copies/phase03c_audit/figure...,/content/drive/MyDrive/AMIR Lab/Research Assis...,118907
4,phase03c_audit,reference_figure_copies,/content/drive/MyDrive/AMIR Lab/Research Assis...,figures/reference_copies/phase03c_audit/figure...,/content/drive/MyDrive/AMIR Lab/Research Assis...,229779
5,phase03c_audit,reports,/content/drive/MyDrive/AMIR Lab/Research Assis...,reports/phase03c_audit/reports/phase03c_clean_...,/content/drive/MyDrive/AMIR Lab/Research Assis...,3818
6,phase03c_audit,reports,/content/drive/MyDrive/AMIR Lab/Research Assis...,reports/phase03c_audit/reports/phase03c_clean_...,/content/drive/MyDrive/AMIR Lab/Research Assis...,1768
7,phase03c_audit,tables,/content/drive/MyDrive/AMIR Lab/Research Assis...,tables/phase03c_audit/tables/phase03c_clean_an...,/content/drive/MyDrive/AMIR Lab/Research Assis...,3846159
8,phase03c_audit,tables,/content/drive/MyDrive/AMIR Lab/Research Assis...,tables/phase03c_audit/tables/phase03c_clean_di...,/content/drive/MyDrive/AMIR Lab/Research Assis...,4110
9,phase03c_audit,tables,/content/drive/MyDrive/AMIR Lab/Research Assis...,tables/phase03c_audit/tables/phase03c_clean_in...,/content/drive/MyDrive/AMIR Lab/Research Assis...,247


Planned artifacts: 698
category
evidence                   326
reports                    241
tables                      61
source_figure_copies        43
reference_figure_copies     27


## Copy Artifacts Into Release Folder


In [ ]:
def sha256_file(path: Path, chunk_size: int = 1024 * 1024) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for chunk in iter(lambda: handle.read(chunk_size), b""):
            digest.update(chunk)
    return digest.hexdigest()


RELEASE_BASE_DIR.mkdir(parents=True, exist_ok=True)

if OUTPUT_ROOT.exists() and any(OUTPUT_ROOT.iterdir()) and ARCHIVE_EXISTING_OUTPUT:
    archive_root = RELEASE_BASE_DIR.parent / f"{RELEASE_BASE_DIR.name}_archived_{RUN_ID}"
    print(f"Existing non-empty release folder found. Archiving to: {archive_root}")
    shutil.move(str(OUTPUT_ROOT), str(archive_root))

for subdir in CATEGORY_DIRS.values():
    (OUTPUT_ROOT / subdir).mkdir(parents=True, exist_ok=True)

manifest_rows = []
for row in copy_plan:
    src = row["source_path"]
    dst = row["destination_path"]
    dst.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(src, dst)
    manifest_rows.append(
        {
            "source_key": row["source_key"],
            "category": row["category"],
            "source_path": str(src),
            "destination_relative": str(row["destination_relative"]),
            "destination_path": str(dst),
            "source_size_bytes": src.stat().st_size,
            "destination_size_bytes": dst.stat().st_size,
            "source_sha256": sha256_file(src),
            "destination_sha256_before_rewrite": sha256_file(dst),
            "rewrite_count": 0,
            "rewrite_mode": "not_text_or_not_rewritten",
            "visual_review_required": False,
            "notes": "",
        }
    )

print(f"Copied {len(manifest_rows)} artifacts into {OUTPUT_ROOT}")


Copied 698 artifacts into /content/drive/MyDrive/AMIR Lab/Research Assistant Intern/research_01_brain_tumor_VQA_medical_imaging/phase_7_paper_ready_augr_vqa_release/btumqa_225k_clean_metadata_four_seeds/results


## Rewrite Labels In Copied Text Artifacts


In [ ]:
TEXT_SUFFIXES = {".csv", ".json", ".jsonl", ".md", ".txt"}


def rewrite_csv(path: Path) -> int:
    df = pd.read_csv(path)
    total = 0
    for col in df.columns:
        if is_provenance_key(col):
            continue
        if df[col].dtype == object:
            def repl(value: Any) -> Any:
                nonlocal total
                if isinstance(value, str):
                    new_value, hits = apply_display_replacements(value)
                    total += hits
                    return new_value
                return value

            df[col] = df[col].map(repl)
    df.to_csv(path, index=False)
    return total


def rewrite_json_obj(obj: Any, key: str = "") -> tuple[Any, int]:
    if isinstance(obj, dict):
        total = 0
        out = {}
        for k, v in obj.items():
            new_v, hits = rewrite_json_obj(v, str(k))
            out[k] = new_v
            total += hits
        return out, total
    if isinstance(obj, list):
        total = 0
        out = []
        for item in obj:
            new_item, hits = rewrite_json_obj(item, key)
            out.append(new_item)
            total += hits
        return out, total
    if isinstance(obj, str) and not is_provenance_key(key):
        new_text, hits = apply_display_replacements(obj)
        return new_text, hits
    return obj, 0


def rewrite_text_file(path: Path) -> tuple[int, str]:
    suffix = path.suffix.lower()
    if suffix == ".csv":
        return rewrite_csv(path), "csv_structured_skip_provenance_columns"
    if suffix == ".json":
        data = json.loads(path.read_text(encoding="utf-8"))
        new_data, hits = rewrite_json_obj(data)
        path.write_text(json.dumps(new_data, indent=2, ensure_ascii=False), encoding="utf-8")
        return hits, "json_structured_skip_provenance_keys"
    if suffix == ".jsonl":
        hits_total = 0
        out_lines = []
        for line in path.read_text(encoding="utf-8").splitlines():
            if not line.strip():
                out_lines.append(line)
                continue
            data = json.loads(line)
            new_data, hits = rewrite_json_obj(data)
            hits_total += hits
            out_lines.append(json.dumps(new_data, ensure_ascii=False))
        path.write_text("\n".join(out_lines) + "\n", encoding="utf-8")
        return hits_total, "jsonl_structured_skip_provenance_keys"
    text = path.read_text(encoding="utf-8", errors="replace")
    new_text, hits = apply_display_replacements(text)
    path.write_text(new_text, encoding="utf-8")
    return hits, "plain_text_rewrite"


destination_to_index = {row["destination_path"]: i for i, row in enumerate(manifest_rows)}
for row in manifest_rows:
    dst = Path(row["destination_path"])
    if dst.suffix.lower() not in TEXT_SUFFIXES:
        continue
    hits, mode = rewrite_text_file(dst)
    row["rewrite_count"] = hits
    row["rewrite_mode"] = mode
    row["destination_size_bytes"] = dst.stat().st_size
    row["destination_sha256_after_rewrite"] = sha256_file(dst)

for row in manifest_rows:
    row.setdefault("destination_sha256_after_rewrite", row["destination_sha256_before_rewrite"])

rewritten = sum(1 for row in manifest_rows if row["rewrite_count"] > 0)
print(f"Text artifacts rewritten: {rewritten}")
print(f"Total replacement hits: {sum(row['rewrite_count'] for row in manifest_rows)}")


Text artifacts rewritten: 42
Total replacement hits: 1216590


## Regenerate Strict Paper-Ready Result Figures


In [ ]:
GENERATED_FIG_DIR = OUTPUT_ROOT / "figures" / "regenerated_paper_ready"
GENERATED_FIG_DIR.mkdir(parents=True, exist_ok=True)


def find_dest_file(name: str, category: str | None = None) -> Path | None:
    matches = []
    for row in manifest_rows:
        path = Path(row["destination_path"])
        if path.name != name:
            continue
        if category and row.get("category") != category:
            continue
        matches.append(path)
    return matches[0] if matches else None


generated_figures = []
figure_regeneration_rows = []


JOURNAL_COLORS = {
    "augr": "#0072B2",
    "qcur": "#009E73",
    "raw": "#D55E00",
    "posthoc": "#CC79A7",
    "cnn": "#4C78A8",
    "transformer": "#7A5195",
    "vlm": "#00A6A6",
    "diagnostic": "#7F7F7F",
    "metric_1": "#0072B2",
    "metric_2": "#E69F00",
    "metric_3": "#009E73",
    "metric_4": "#CC79A7",
    "metric_5": "#56B4E9",
    "metric_6": "#D55E00",
}

MODEL_LABEL_REPLACEMENTS = [
    ("QAdp-DG-PRUGTM + Q-CUR", "AUGR-VQA"),
    ("QAdp PRUGTM Hybrid + Q-CUR", "AUGR-VQA"),
    ("QAdp-PRUGTM-Hybrid + Q-CUR", "AUGR-VQA"),
    ("qadp_dg_prugtm_qcur", "AUGR-VQA"),
    ("qadp_prugtm_hybrid_qcur", "AUGR-VQA"),
    ("augr_vqa", "AUGR-VQA"),
    ("AUGR VQA", "AUGR-VQA"),
    ("QAdp-PRUGTM-Hybrid", "QAdp-DG-PRUGTM"),
    ("QAdp PRUGTM Hybrid", "QAdp-DG-PRUGTM"),
    ("qadp_dg_prugtm", "QAdp-DG-PRUGTM"),
    ("qadp_prugtm_hybrid_clean", "QAdp-DG-PRUGTM"),
    ("qadp_prugtm_hybrid", "QAdp-DG-PRUGTM"),
    ("selected_model", "AUGR-VQA"),
    ("DenseNet121 MLP", "DenseNet121"),
    ("ResNet50 MLP", "ResNet50"),
    ("EfficientNet-B0 MLP", "EfficientNet-B0"),
    ("ViT-B/16 MLP", "ViT-B/16"),
    ("Swin-T MLP", "Swin-T"),
    ("BiomedCLIP dual-view naturalized Global-23", "BiomedCLIP-A3"),
    ("Native frozen BiomedCLIP", "BiomedCLIP-A3"),
    ("BiomedCLIP frozen-feature supervised", "BiomedCLIP-A4"),
    ("Supervised frozen BiomedCLIP", "BiomedCLIP-A4"),
    ("BiomedCLIP Frozen Question+Image MLP", "BiomedCLIP-A4"),
    ("BiomedCLIP Frozen Image-Only MLP", "BiomedCLIP-A4 Image-only"),
    ("BiomedCLIP Naturalized Global-23 Ensemble", "BiomedCLIP-A3"),
    ("DenseNet121 Question+Image MLP", "DenseNet121"),
    ("ResNet50 Question+Image MLP", "ResNet50"),
    ("EfficientNet-B0 Question+Image MLP", "EfficientNet-B0"),
    ("ViT-B/16 Question+Image MLP", "ViT-B/16"),
    ("Swin-T Question+Image MLP", "Swin-T"),
    ("DenseNet121 Image-Only MLP", "DenseNet121 Image-only"),
    ("ResNet50 Image-Only MLP", "ResNet50 Image-only"),
    ("EfficientNet-B0 Image-Only MLP", "EfficientNet-B0 Image-only"),
    ("ViT-B/16 Image-Only MLP", "ViT-B/16 Image-only"),
    ("Swin-T Image-Only MLP", "Swin-T Image-only"),
]

METRIC_LABELS = {
    "test_accuracy_mean": "Accuracy",
    "test_macro_f1_mean": "Macro-F1",
    "test_weighted_f1_mean": "Weighted-F1",
    "accuracy": "Accuracy",
    "macro_f1": "Macro-F1",
    "weighted_f1": "Weighted-F1",
    "ece": "ECE",
    "brier": "Brier",
    "nll": "NLL",
    "aurc": "AURC",
    "ece__mean": "ECE",
    "brier__mean": "Brier",
    "nll__mean": "NLL",
    "aurc__mean": "AURC",
    "ece_mean": "ECE",
    "brier_mean": "Brier",
    "nll_mean": "NLL",
    "aurc_mean": "AURC",
    "raw_confidence_mean": "Raw confidence",
    "qcur_confidence_mean": "Q-CUR confidence",
    "qadp_error_rate": "Error rate",
    "qadp_minus_best_clean_safe_macro_f1": "Macro-F1 gap",
    "qadp_minus_reference": "Macro-F1 gain",
    "observed_mean": "Observed mean",
    "observed_difference": "Observed difference",
    "macro_f1_mean": "Macro-F1",
    "ece_gain": "ECE gain",
    "brier_gain": "Brier gain",
    "nll_gain": "NLL gain",
    "aurc_gain": "AURC gain",
}

PAPER_TITLES = {
    "phase03d_clean_augr_vqa_confidence_by_correctness.png": "Confidence By Answer Correctness",
    "phase03d_clean_augr_vqa_lowest_shortcut_gap_slices.png": "Lowest Shortcut Gap Slices",
    "phase03d_clean_augr_vqa_error_concentration.png": "AUGR-VQA Error Concentration",
    "phase03d_clean_augr_vqa_review_flag_error_rate.png": "Q-CUR Review-Flag Error Rate",
    "phase03d_clean_augr_vqa_shortcut_vs_model_macro_f1.png": "Shortcut Diagnostic Comparison",
    "phase03g_clean_input_ablation_macro_f1_comparison.png": "Input Ablation Macro-F1",
    "phase03g_clean_input_ablation_metric_panel.png": "Input Ablation Metrics",
    "phase03g_augr_vqa_macro_f1_gain_over_input_ablations.png": "QAdp-DG-PRUGTM Gain Over Input Ablations",
    "phase03g_question_only_strongest_slices.png": "Question-Only Strongest Slices",
    "phase5b_augr_vqa_answer_backbone_comparison.png": "Answer Backbone Comparison",
    "phase5b_augr_vqa_ablation_comparison.png": "Ablation Comparison",
    "phase5b_augr_vqa_calibration_metrics_comparison.png": "Q-CUR Reliability Metrics",
    "phase5b_augr_vqa_ece_slice_heatmap.png": "Slice-Level ECE",
    "phase5b_augr_vqa_qcur_gain_summary.png": "Q-CUR Reliability Gain",
    "phase5b_augr_vqa_reviewer_summary_dashboard.png": "AUGR-VQA Result Summary",
    "no_ugtm_qcur_reliability_summary.png": "No-UGTM Reliability Summary",
    "no_ugtm_qcur_risk_coverage_summary.png": "No-UGTM Risk-Coverage Summary",
    "prugtm_hybrid_qcur_reliability_summary.png": "PRUGTM-Hybrid Reliability Summary",
    "prugtm_hybrid_qcur_risk_coverage_summary.png": "PRUGTM-Hybrid Risk-Coverage Summary",
    "augr_vqa_qcur_reliability_summary.png": "AUGR-VQA Reliability Summary",
    "augr_vqa_qcur_risk_coverage_summary.png": "AUGR-VQA Risk-Coverage Summary",
    "ugtm_qgca_qcur_reliability_summary.png": "UGTM-QGCA Reliability Summary",
    "ugtm_qgca_qcur_risk_coverage_summary.png": "UGTM-QGCA Risk-Coverage Summary",
    "phase5c_augr_vqa_bootstrap_ci_answer_metrics.png": "Bootstrap Confidence Intervals",
    "phase5c_augr_vqa_bootstrap_pairwise_difference_ci.png": "Paired Difference Confidence Intervals",
    "phase5c_augr_vqa_reviewer_summary_dashboard.png": "Statistical Robustness Summary",
    "phase5c_augr_vqa_slice_aurc_qcur_gain_heatmap.png": "Slice-Level Q-CUR AURC Gain",
    "phase5c_augr_vqa_slice_brier_qcur_gain_heatmap.png": "Slice-Level Q-CUR Brier Gain",
    "phase5c_augr_vqa_slice_ece_qcur_gain_heatmap.png": "Slice-Level Q-CUR ECE Gain",
    "phase5c_augr_vqa_slice_macro_f1_heatmap.png": "Slice-Level Macro-F1",
    "phase5c_augr_vqa_slice_nll_qcur_gain_heatmap.png": "Slice-Level Q-CUR NLL Gain",
    "phase06a_augr_vqa_all_models_ablation_comparison.png": "Ablation Comparison Across Models",
    "phase06a_augr_vqa_all_models_calibration_reliability.png": "Calibration And Reliability Comparison",
    "phase06a_augr_vqa_all_models_evaluation_metrics.png": "Same-Dataset Baseline Comparison",
    "phase06a_augr_vqa_all_models_statistical_robustness.png": "Statistical Robustness Comparison",
    "phase06a_augr_vqa_models_unified_results_dashboard_summary.png": "Unified Results Summary",
    "phase06a_augr_vqa_table_ablation_comparison.png": "Ablation Comparison Table",
    "phase06a_augr_vqa_table_calibration_reliability.png": "Calibration Reliability Table",
    "phase06a_augr_vqa_table_main_model_comparison.png": "Main Model Comparison Table",
    "phase06a_augr_vqa_table_statistical_robustness.png": "Statistical Robustness Table",
    "phase06a_augr_vqa_unified_calibration_reliability_panel.png": "Unified Calibration Reliability",
    "phase06a_augr_vqa_unified_dashboard_summary.png": "Unified Result Summary",
    "phase06a_augr_vqa_unified_tiered_model_comparison.png": "Tiered Same-Dataset Model Comparison",
}

def paper_label(value: Any) -> Any:
    if not isinstance(value, str):
        return value
    text = apply_display_replacements(value)[0].strip()
    for old, new in MODEL_LABEL_REPLACEMENTS:
        if old in text:
            text = text.replace(old, new)
    text = re.sub(r"\s+", " ", text).strip()
    return text


def paper_metric_label(value: str) -> str:
    return METRIC_LABELS.get(value, value.replace("__mean", "").replace("_mean", "").replace("_", " ").title())


def paper_title_for(out_name: str, fallback: str = "") -> str:
    title = PAPER_TITLES.get(out_name, fallback or out_name)
    title = title.replace(".png", "").replace("_", " ")
    title = re.sub(r"\b[Pp]hase\s*0?\d+[A-Za-z]?\b", "", title)
    title = re.sub(r"\bphase0?\d+[a-z]?\b", "", title, flags=re.IGNORECASE)
    title = title.replace("clean metadata four seeds", "")
    title = title.replace("Clean Metadata Four Seeds", "")
    title = re.sub(r"\s+", " ", title).strip(" -_")
    return title


def apply_paper_plot_style() -> None:
    plt.rcParams.update({
        "figure.facecolor": "white",
        "axes.facecolor": "white",
        "axes.edgecolor": "#333333",
        "axes.labelcolor": "#222222",
        "xtick.color": "#222222",
        "ytick.color": "#222222",
        "font.size": 10,
        "axes.titlesize": 12,
        "axes.titleweight": "bold",
        "axes.labelsize": 10,
        "legend.fontsize": 9,
    })


def color_for_label(label: Any) -> str:
    text = str(label).lower()
    if "augr" in text or "qadp-dg" in text:
        return JOURNAL_COLORS["augr"]
    if "q-cur" in text or "qcur" in text:
        return JOURNAL_COLORS["qcur"]
    if "raw" in text:
        return JOURNAL_COLORS["raw"]
    if "posthoc" in text or "post-hoc" in text or "calibration" in text:
        return JOURNAL_COLORS["posthoc"]
    if any(token in text for token in ["densenet", "resnet", "efficientnet"]):
        return JOURNAL_COLORS["cnn"]
    if any(token in text for token in ["vit", "swin", "transformer", "attention"]):
        return JOURNAL_COLORS["transformer"]
    if any(token in text for token in ["biomedclip", "llava", "medgemma", "vlm"]):
        return JOURNAL_COLORS["vlm"]
    if any(token in text for token in ["shortcut", "majority", "question", "descriptor", "forbidden", "baseline"]):
        return JOURNAL_COLORS["diagnostic"]
    palette = [JOURNAL_COLORS[f"metric_{i}"] for i in range(1, 7)]
    return palette[abs(hash(str(label))) % len(palette)]


def metric_palette(metric_cols: list[str]) -> list[str]:
    palette = [JOURNAL_COLORS[f"metric_{i}"] for i in range(1, 7)]
    return [palette[index % len(palette)] for index, _ in enumerate(metric_cols)]


def sanitize_df_labels(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    for col in out.columns:
        if out[col].dtype == object:
            out[col] = out[col].map(paper_label)
    return out


def finalize_plot(out_name: str) -> Path:
    out_path = GENERATED_FIG_DIR / out_name
    plt.tight_layout()
    plt.savefig(out_path, dpi=320, bbox_inches="tight")
    plt.close()
    generated_figures.append(out_path)
    return out_path


def save_metric_bar(df: pd.DataFrame, label_col: str, metric_cols: list[str], title: str, out_name: str) -> Path:
    apply_paper_plot_style()
    df = sanitize_df_labels(df)
    plot_df = df[[label_col] + metric_cols].copy()
    plot_df = plot_df.rename(columns={col: paper_metric_label(col) for col in metric_cols})
    metric_labels = [paper_metric_label(col) for col in metric_cols]
    ax = plot_df.set_index(label_col)[metric_labels].plot(
        kind="barh",
        figsize=(10, max(4, 0.42 * df.shape[0] + 2)),
        color=metric_palette(metric_cols),
        width=0.74,
    )
    ax.set_title(paper_title_for(out_name, title))
    ax.set_xlabel("Score")
    ax.set_ylabel("")
    ax.grid(axis="x", alpha=0.22, linestyle="-", linewidth=0.6)
    ax.legend(frameon=False, loc="best")
    return finalize_plot(out_name)


def save_single_bar(df: pd.DataFrame, label_col: str, value_col: str, title: str, out_name: str, xlabel: str = "Value") -> Path:
    apply_paper_plot_style()
    df = sanitize_df_labels(df)
    plot_df = df[[label_col, value_col]].dropna().copy()
    plot_df = plot_df.sort_values(value_col, ascending=True)
    colors = [color_for_label(label) for label in plot_df[label_col]]
    plt.figure(figsize=(9, max(4, 0.38 * len(plot_df) + 1.5)))
    plt.barh(plot_df[label_col].astype(str), plot_df[value_col], color=colors)
    plt.title(paper_title_for(out_name, title))
    plt.xlabel(paper_metric_label(xlabel if xlabel != "Value" else value_col))
    plt.ylabel("")
    plt.grid(axis="x", alpha=0.22, linestyle="-", linewidth=0.6)
    return finalize_plot(out_name)


def save_errorbar(df: pd.DataFrame, label_col: str, value_col: str, low_col: str, high_col: str, title: str, out_name: str) -> Path:
    apply_paper_plot_style()
    df = sanitize_df_labels(df)
    plot_df = df[[label_col, value_col, low_col, high_col]].dropna().copy()
    y = range(len(plot_df))
    x = plot_df[value_col].astype(float)
    xerr = [x - plot_df[low_col].astype(float), plot_df[high_col].astype(float) - x]
    plt.figure(figsize=(9, max(4, 0.38 * len(plot_df) + 1.5)))
    colors = [color_for_label(label) for label in plot_df[label_col]]
    for idx, color in enumerate(colors):
        plt.errorbar(x.iloc[idx], idx, xerr=[[xerr[0].iloc[idx]], [xerr[1].iloc[idx]]], fmt="o", color=color, ecolor=color, capsize=3)
    plt.yticks(list(y), plot_df[label_col].astype(str))
    plt.title(paper_title_for(out_name, title))
    plt.xlabel(paper_metric_label(value_col))
    plt.grid(axis="x", alpha=0.22, linestyle="-", linewidth=0.6)
    return finalize_plot(out_name)


def save_heatmap(df: pd.DataFrame, index_col: str, column_col: str, value_col: str, title: str, out_name: str) -> Path:
    apply_paper_plot_style()
    df = sanitize_df_labels(df)
    pivot = df.pivot_table(index=index_col, columns=column_col, values=value_col, aggfunc="mean")
    plt.figure(figsize=(max(8, 0.8 * len(pivot.columns) + 3), max(4, 0.35 * len(pivot.index) + 2)))
    plt.imshow(pivot.fillna(0).values, aspect="auto", cmap="cividis")
    plt.colorbar(label=paper_metric_label(value_col))
    plt.xticks(range(len(pivot.columns)), pivot.columns.astype(str), rotation=45, ha="right")
    plt.yticks(range(len(pivot.index)), pivot.index.astype(str))
    plt.title(paper_title_for(out_name, title))
    return finalize_plot(out_name)


def save_table_image(df: pd.DataFrame, title: str, out_name: str, max_rows: int = 12) -> Path:
    apply_paper_plot_style()
    df = sanitize_df_labels(df).head(max_rows).copy()
    plt.figure(figsize=(min(18, max(8, len(df.columns) * 1.4)), max(3, len(df) * 0.42 + 1.8)))
    plt.axis("off")
    plt.title(paper_title_for(out_name, title), pad=14, fontweight="bold")
    table = plt.table(cellText=df.values, colLabels=df.columns, loc="center", cellLoc="center")
    table.auto_set_font_size(False)
    table.set_fontsize(8)
    table.scale(1, 1.35)
    return finalize_plot(out_name)


FIGURE_REGENERATION_REGISTRY = [
    ("phase03d_clean_confidence_correctness_histograms.png", "phase03d_clean_augr_vqa_confidence_by_correctness.png", "phase03d_clean_augr_vqa_confidence_by_correctness.csv", "phase03d_confidence"),
    ("phase03d_clean_lowest_qadp_shortcut_gap_slices.png", "phase03d_clean_augr_vqa_lowest_shortcut_gap_slices.png", "phase03d_clean_augr_vqa_shortcut_gap_by_slice.csv", "single_bar:qadp_minus_best_clean_safe_macro_f1"),
    ("phase03d_clean_qadp_error_concentration.png", "phase03d_clean_augr_vqa_error_concentration.png", "phase03d_clean_augr_vqa_error_concentration.csv", "single_bar:qadp_error_rate"),
    ("phase03d_clean_review_flag_error_rate.png", "phase03d_clean_augr_vqa_review_flag_error_rate.png", "phase03d_clean_review_flag_usefulness.csv", "single_bar:qadp_error_rate"),
    ("phase03d_clean_shortcut_vs_qadp_macro_f1.png", "phase03d_clean_augr_vqa_shortcut_vs_model_macro_f1.png", "phase03d_clean_shortcut_vs_augr_vqa_overall_metrics.csv", "single_bar:macro_f1"),
    ("phase03g_clean_input_ablation_macro_f1_comparison.png", "phase03g_clean_input_ablation_macro_f1_comparison.png", "phase03g_clean_input_ablation_aggregated_metrics.csv", "single_bar:test_macro_f1_mean"),
    ("phase03g_clean_input_ablation_metric_panel.png", "phase03g_clean_input_ablation_metric_panel.png", "phase03g_clean_input_ablation_aggregated_metrics.csv", "metrics:test_accuracy_mean,test_macro_f1_mean,test_weighted_f1_mean"),
    ("phase03g_qadp_macro_f1_gain_over_input_ablations.png", "phase03g_augr_vqa_macro_f1_gain_over_input_ablations.png", "phase03g_clean_input_ablation_qadp_gap_metrics.csv", "single_bar:qadp_minus_reference"),
    ("phase03g_question_only_strongest_slices.png", "phase03g_question_only_strongest_slices.png", "phase03g_clean_input_ablation_slice_aggregated_metrics.csv", "single_bar:macro_f1_mean"),
    ("phase5b_clean_metadata_four_seeds_answer_backbone_comparison.png", "phase5b_augr_vqa_answer_backbone_comparison.png", "phase5b_augr_vqa_answer_backbone_table.csv", "metrics:test_accuracy_mean,test_macro_f1_mean,test_weighted_f1_mean"),
    ("phase5b_clean_metadata_four_seeds_ablation_comparison.png", "phase5b_augr_vqa_ablation_comparison.png", "phase5b_augr_vqa_ablation_comparison_table.csv", "metrics:test_accuracy_mean,test_macro_f1_mean,test_weighted_f1_mean"),
    ("phase5b_clean_metadata_four_seeds_calibration_metrics_comparison.png", "phase5b_augr_vqa_calibration_metrics_comparison.png", "phase5b_augr_vqa_calibration_aggregated.csv", "metrics:ece__mean,brier__mean,nll__mean,aurc__mean"),
    ("phase5b_clean_metadata_four_seeds_ece_heatmap.png", "phase5b_augr_vqa_ece_slice_heatmap.png", "phase5b_augr_vqa_key_slice_calibration.csv", "heatmap:backbone,slice_value,ece__mean"),
    ("phase5b_clean_metadata_four_seeds_qcur_gain_summary.png", "phase5b_augr_vqa_qcur_gain_summary.png", "phase5b_augr_vqa_calibration_aggregated.csv", "metrics:ece__mean,brier__mean,nll__mean,aurc__mean"),
    ("phase5b_clean_metadata_four_seeds_reviewer_summary_dashboard.png", "phase5b_augr_vqa_reviewer_summary_dashboard.png", "phase5b_augr_vqa_answer_backbone_table.csv", "metrics:test_accuracy_mean,test_macro_f1_mean,test_weighted_f1_mean"),
    ("no_ugtm_clean_four_seeds_reliability_diagram.png", "no_ugtm_qcur_reliability_summary.png", "phase5b_augr_vqa_calibration_aggregated.csv", "filter_metrics:backbone,No-UGTM,ece__mean,brier__mean,nll__mean,aurc__mean"),
    ("no_ugtm_clean_four_seeds_risk_coverage.png", "no_ugtm_qcur_risk_coverage_summary.png", "phase5b_augr_vqa_calibration_aggregated.csv", "filter_single:backbone,No-UGTM,aurc__mean"),
    ("prugtm_hybrid_clean_four_seeds_reliability_diagram.png", "prugtm_hybrid_qcur_reliability_summary.png", "phase5b_augr_vqa_calibration_aggregated.csv", "filter_metrics:backbone,PRUGTM-Hybrid,ece__mean,brier__mean,nll__mean,aurc__mean"),
    ("prugtm_hybrid_clean_four_seeds_risk_coverage.png", "prugtm_hybrid_qcur_risk_coverage_summary.png", "phase5b_augr_vqa_calibration_aggregated.csv", "filter_single:backbone,PRUGTM-Hybrid,aurc__mean"),
    ("qadp_prugtm_hybrid_clean_four_seeds_reliability_diagram.png", "augr_vqa_qcur_reliability_summary.png", "phase5b_augr_vqa_calibration_aggregated.csv", "filter_metrics:backbone,QAdp-DG-PRUGTM,ece__mean,brier__mean,nll__mean,aurc__mean"),
    ("qadp_prugtm_hybrid_clean_four_seeds_risk_coverage.png", "augr_vqa_qcur_risk_coverage_summary.png", "phase5b_augr_vqa_calibration_aggregated.csv", "filter_single:backbone,QAdp-DG-PRUGTM,aurc__mean"),
    ("ugtm_qgca_clean_four_seeds_reliability_diagram.png", "ugtm_qgca_qcur_reliability_summary.png", "phase5b_augr_vqa_calibration_aggregated.csv", "filter_metrics:backbone,UGTM-QGCA,ece__mean,brier__mean,nll__mean,aurc__mean"),
    ("ugtm_qgca_clean_four_seeds_risk_coverage.png", "ugtm_qgca_qcur_risk_coverage_summary.png", "phase5b_augr_vqa_calibration_aggregated.csv", "filter_single:backbone,UGTM-QGCA,aurc__mean"),
    ("phase5c_clean_metadata_four_seeds_bootstrap_ci_answer_metrics.png", "phase5c_augr_vqa_bootstrap_ci_answer_metrics.png", "phase5c_augr_vqa_bootstrap_ci_global_metrics.csv", "errorbar:metric,observed_mean,ci_lower,ci_upper"),
    ("phase5c_clean_metadata_four_seeds_bootstrap_pairwise_difference_ci.png", "phase5c_augr_vqa_bootstrap_pairwise_difference_ci.png", "phase5c_augr_vqa_bootstrap_pairwise_differences.csv", "errorbar:comparison_id,observed_difference,ci_lower,ci_upper"),
    ("phase5c_clean_metadata_four_seeds_reviewer_summary_dashboard.png", "phase5c_augr_vqa_reviewer_summary_dashboard.png", "phase5c_augr_vqa_bootstrap_ci_global_metrics.csv", "errorbar:metric,observed_mean,ci_lower,ci_upper"),
    ("phase5c_clean_metadata_four_seeds_slice_aurc_qcur_gain_heatmap.png", "phase5c_augr_vqa_slice_aurc_qcur_gain_heatmap.png", "phase5c_augr_vqa_key_slice_qcur_gain_summary.csv", "heatmap:backbone,slice_value,aurc_gain"),
    ("phase5c_clean_metadata_four_seeds_slice_brier_qcur_gain_heatmap.png", "phase5c_augr_vqa_slice_brier_qcur_gain_heatmap.png", "phase5c_augr_vqa_key_slice_qcur_gain_summary.csv", "heatmap:backbone,slice_value,brier_gain"),
    ("phase5c_clean_metadata_four_seeds_slice_ece_qcur_gain_heatmap.png", "phase5c_augr_vqa_slice_ece_qcur_gain_heatmap.png", "phase5c_augr_vqa_key_slice_qcur_gain_summary.csv", "heatmap:backbone,slice_value,ece_gain"),
    ("phase5c_clean_metadata_four_seeds_slice_macro_f1_heatmap.png", "phase5c_augr_vqa_slice_macro_f1_heatmap.png", "phase5c_augr_vqa_key_slice_answer_comparison.csv", "heatmap:backbone,slice_value,macro_f1_mean"),
    ("phase5c_clean_metadata_four_seeds_slice_nll_qcur_gain_heatmap.png", "phase5c_augr_vqa_slice_nll_qcur_gain_heatmap.png", "phase5c_augr_vqa_key_slice_qcur_gain_summary.csv", "heatmap:backbone,slice_value,nll_gain"),
    ("phase06a_all_models_ablation_comparison.png", "phase06a_augr_vqa_all_models_ablation_comparison.png", "phase06a_augr_vqa_unified_ablation_comparison.csv", "metrics:Accuracy,Macro-F1,Weighted-F1"),
    ("phase06a_all_models_calibration_reliability.png", "phase06a_augr_vqa_all_models_calibration_reliability.png", "phase06a_augr_vqa_unified_calibration_reliability_comparison.csv", "metrics:ece,brier,nll,aurc"),
    ("phase06a_all_models_evaluation_metrics.png", "phase06a_augr_vqa_all_models_evaluation_metrics.png", "phase06a_augr_vqa_unified_main_paper_model_comparison.csv", "metrics:accuracy,macro_f1,weighted_f1"),
    ("phase06a_all_models_statistical_robustness.png", "phase06a_augr_vqa_all_models_statistical_robustness.png", "phase06a_augr_vqa_unified_statistical_robustness_comparison.csv", "errorbar:metric,observed_mean,ci_lower,ci_upper"),
    ("phase06a_models_unified_results_dashboard_summary.png", "phase06a_augr_vqa_models_unified_results_dashboard_summary.png", "phase06a_augr_vqa_unified_main_paper_model_comparison.csv", "metrics:accuracy,macro_f1,weighted_f1"),
    ("phase06a_table_ablation_comparison.png", "phase06a_augr_vqa_table_ablation_comparison.png", "phase06a_augr_vqa_unified_ablation_comparison.csv", "table"),
    ("phase06a_table_calibration_reliability.png", "phase06a_augr_vqa_table_calibration_reliability.png", "phase06a_augr_vqa_unified_calibration_reliability_comparison.csv", "table"),
    ("phase06a_table_main_model_comparison.png", "phase06a_augr_vqa_table_main_model_comparison.png", "phase06a_augr_vqa_unified_main_paper_model_comparison.csv", "table"),
    ("phase06a_table_statistical_robustness.png", "phase06a_augr_vqa_table_statistical_robustness.png", "phase06a_augr_vqa_unified_statistical_robustness_comparison.csv", "table"),
    ("phase06a_unified_calibration_reliability_panel.png", "phase06a_augr_vqa_unified_calibration_reliability_panel.png", "phase06a_augr_vqa_unified_calibration_reliability_comparison.csv", "metrics:ece,brier,nll,aurc"),
    ("phase06a_unified_dashboard_summary.png", "phase06a_augr_vqa_unified_dashboard_summary.png", "phase06a_augr_vqa_unified_main_paper_model_comparison.csv", "metrics:accuracy,macro_f1,weighted_f1"),
    ("phase06a_unified_tiered_model_comparison.png", "phase06a_augr_vqa_unified_tiered_model_comparison.png", "phase06a_augr_vqa_unified_main_paper_model_comparison.csv", "metrics:accuracy,macro_f1,weighted_f1"),
]


def label_column_for(df: pd.DataFrame) -> str:
    for col in ["backbone", "model_label", "Model Family", "Variant", "comparison_id", "metric", "slice_value", "review_flag", "qadp_correct", "reference_model_label"]:
        if col in df.columns:
            return col
    return df.columns[0]


def render_registry_figure(entry: tuple[str, str, str, str]) -> dict:
    source_png, out_name, source_csv_name, plot_kind = entry
    source_csv = find_dest_file(source_csv_name)
    if source_csv is None:
        return {
            "source_png": source_png,
            "source_csv": source_csv_name,
            "regenerated_output": "",
            "status": "missing_source_table_blocked",
            "paper_use_allowed": False,
            "notes": "Required copied CSV was not found.",
        }
    df = sanitize_df_labels(pd.read_csv(source_csv))
    out_path = None
    try:
        if plot_kind == "table":
            out_path = save_table_image(df, out_name.replace("_", " ").replace(".png", "").title(), out_name)
        elif plot_kind == "phase03d_confidence":
            tmp = df.copy()
            tmp["correctness"] = tmp["qadp_correct"].map({True: "Correct", False: "Incorrect", "True": "Correct", "False": "Incorrect"}).fillna(tmp["qadp_correct"].astype(str))
            out_path = save_metric_bar(tmp, "correctness", ["raw_confidence_mean", "qcur_confidence_mean"], "AUGR-VQA Confidence By Correctness", out_name)
        elif plot_kind.startswith("single_bar:"):
            value_col = plot_kind.split(":", 1)[1]
            out_path = save_single_bar(df.head(20), label_column_for(df), value_col, out_name.replace("_", " ").replace(".png", "").title(), out_name)
        elif plot_kind.startswith("metrics:"):
            metric_cols = [col for col in plot_kind.split(":", 1)[1].split(",") if col in df.columns]
            out_path = save_metric_bar(df.head(20), label_column_for(df), metric_cols, out_name.replace("_", " ").replace(".png", "").title(), out_name)
        elif plot_kind.startswith("filter_metrics:"):
            filter_col, filter_value, metrics = plot_kind.split(":", 1)[1].split(",", 2)
            subset = df[df[filter_col].astype(str) == filter_value]
            metric_cols = [col for col in metrics.split(",") if col in subset.columns]
            out_path = save_metric_bar(subset, "method" if "method" in subset.columns else label_column_for(subset), metric_cols, out_name.replace("_", " ").replace(".png", "").title(), out_name)
        elif plot_kind.startswith("filter_single:"):
            filter_col, filter_value, value_col = plot_kind.split(":", 1)[1].split(",", 2)
            subset = df[df[filter_col].astype(str) == filter_value]
            out_path = save_single_bar(subset, "method" if "method" in subset.columns else label_column_for(subset), value_col, out_name.replace("_", " ").replace(".png", "").title(), out_name)
        elif plot_kind.startswith("errorbar:"):
            label_col, value_col, low_col, high_col = plot_kind.split(":", 1)[1].split(",", 3)
            out_path = save_errorbar(df.head(25), label_col, value_col, low_col, high_col, out_name.replace("_", " ").replace(".png", "").title(), out_name)
        elif plot_kind.startswith("heatmap:"):
            index_col, column_col, value_col = plot_kind.split(":", 1)[1].split(",", 2)
            out_path = save_heatmap(df, index_col, column_col, value_col, out_name.replace("_", " ").replace(".png", "").title(), out_name)
        else:
            raise ValueError(f"Unknown plot kind: {plot_kind}")
        return {
            "source_png": source_png,
            "source_csv": str(source_csv),
            "regenerated_output": str(out_path),
            "status": "regenerated",
            "paper_use_allowed": True,
            "notes": plot_kind,
        }
    except Exception as exc:
        return {
            "source_png": source_png,
            "source_csv": str(source_csv),
            "regenerated_output": "",
            "status": "missing_source_table_blocked",
            "paper_use_allowed": False,
            "notes": f"{type(exc).__name__}: {exc}",
        }


for entry in FIGURE_REGENERATION_REGISTRY:
    figure_regeneration_rows.append(render_registry_figure(entry))

print(f"Regenerated {len(generated_figures)} strict paper-ready result figures.")
for path in generated_figures:
    print(path)


Regenerated 43 strict paper-ready result figures.
/content/drive/MyDrive/AMIR Lab/Research Assistant Intern/research_01_brain_tumor_VQA_medical_imaging/phase_7_paper_ready_augr_vqa_release/btumqa_225k_clean_metadata_four_seeds/results/figures/regenerated_paper_ready/phase03d_clean_augr_vqa_confidence_by_correctness.png
/content/drive/MyDrive/AMIR Lab/Research Assistant Intern/research_01_brain_tumor_VQA_medical_imaging/phase_7_paper_ready_augr_vqa_release/btumqa_225k_clean_metadata_four_seeds/results/figures/regenerated_paper_ready/phase03d_clean_augr_vqa_lowest_shortcut_gap_slices.png
/content/drive/MyDrive/AMIR Lab/Research Assistant Intern/research_01_brain_tumor_VQA_medical_imaging/phase_7_paper_ready_augr_vqa_release/btumqa_225k_clean_metadata_four_seeds/results/figures/regenerated_paper_ready/phase03d_clean_augr_vqa_error_concentration.png
/content/drive/MyDrive/AMIR Lab/Research Assistant Intern/research_01_brain_tumor_VQA_medical_imaging/phase_7_paper_ready_augr_vqa_release/btu

## Validate Paper-Ready Figure Coverage


In [ ]:
OLD_LABEL_PATTERNS = [
    "QAdp-PRUGTM-Hybrid",
    "qadp_prugtm_hybrid",
    "Adp-PRUGTM-Hybrid",
]

copied_png_names = []
architecture_names = []
for row in manifest_rows:
    dst = Path(row["destination_path"])
    if dst.suffix.lower() not in {".png", ".jpg", ".jpeg", ".svg"}:
        continue
    if row.get("category") == "architecture_figures":
        row["visual_review_required"] = False
        row["notes"] = "Architecture/method figure copied from explicit ARCHITECTURE_FIGURE_SOURCE_DIR; paper-use allowed if manually designed with final labels."
        architecture_names.append(dst.name)
    elif row.get("category") == "source_figure_copies":
        row["visual_review_required"] = True
        row["notes"] = "Source notebook figure copied for traceability only; use regenerated_paper_ready equivalent for paper."
        copied_png_names.append(Path(row["source_path"]).name)
    elif row.get("category") == "reference_figure_copies":
        row["visual_review_required"] = True
        row["notes"] = "Reference/support figure copied for reviewer traceability; not part of strict paper-use regenerated result figures."

registry_source_names = {entry[0] for entry in FIGURE_REGENERATION_REGISTRY}
copied_result_png_names = set(copied_png_names)
missing_registry = sorted(copied_result_png_names - registry_source_names)
unused_registry = sorted(registry_source_names - copied_result_png_names)

for name in missing_registry:
    figure_regeneration_rows.append(
        {
            "source_png": name,
            "source_csv": "",
            "regenerated_output": "",
            "status": "source_copy_only_blocked",
            "paper_use_allowed": False,
            "notes": "No regeneration registry entry for this source PNG.",
        }
    )

for name in architecture_names:
    figure_regeneration_rows.append(
        {
            "source_png": name,
            "source_csv": "",
            "regenerated_output": str(OUTPUT_ROOT / "figures" / "architecture" / name),
            "status": "architecture_copy",
            "paper_use_allowed": True,
            "notes": "Manual architecture figure; not regenerated from CSV.",
        }
    )

if not architecture_names and ARCHITECTURE_FIGURE_SOURCE_DIR is None:
    figure_regeneration_rows.append(
        {
            "source_png": "",
            "source_csv": "",
            "regenerated_output": "",
            "status": "architecture_not_provided_in_colab",
            "paper_use_allowed": False,
            "notes": "Architecture figures are local-only unless uploaded/copied and ARCHITECTURE_FIGURE_SOURCE_DIR is set.",
        }
    )

figure_regeneration_df = pd.DataFrame(figure_regeneration_rows)
display(figure_regeneration_df)

blocking_statuses = {"source_copy_only_blocked", "missing_source_table_blocked"}
blocked = figure_regeneration_df[figure_regeneration_df["status"].isin(blocking_statuses)]
FIGURE_COVERAGE_PASSED = not (REQUIRE_ALL_RESULT_FIGURES_REGENERATED and not blocked.empty)
if REQUIRE_ALL_RESULT_FIGURES_REGENERATED and not blocked.empty:
    print("ERROR: strict figure regeneration has blocked rows. The manifest cell will still write diagnostics, but the release is not complete.")
    display(blocked)
else:
    print("Strict figure coverage validation passed.")

if unused_registry:
    print("Registry entries not found among copied source PNGs; this can happen if a source figure is absent in the current Drive export.")
    print(unused_registry)

print(f"FIGURE_COVERAGE_PASSED: {FIGURE_COVERAGE_PASSED}")


,source_png,source_csv,regenerated_output,status,paper_use_allowed,notes
0,phase03d_clean_confidence_correctness_histogra...,/content/drive/MyDrive/AMIR Lab/Research Assis...,/content/drive/MyDrive/AMIR Lab/Research Assis...,regenerated,True,phase03d_confidence
1,phase03d_clean_lowest_qadp_shortcut_gap_slices...,/content/drive/MyDrive/AMIR Lab/Research Assis...,/content/drive/MyDrive/AMIR Lab/Research Assis...,regenerated,True,single_bar:qadp_minus_best_clean_safe_macro_f1
2,phase03d_clean_qadp_error_concentration.png,/content/drive/MyDrive/AMIR Lab/Research Assis...,/content/drive/MyDrive/AMIR Lab/Research Assis...,regenerated,True,single_bar:qadp_error_rate
3,phase03d_clean_review_flag_error_rate.png,/content/drive/MyDrive/AMIR Lab/Research Assis...,/content/drive/MyDrive/AMIR Lab/Research Assis...,regenerated,True,single_bar:qadp_error_rate
4,phase03d_clean_shortcut_vs_qadp_macro_f1.png,/content/drive/MyDrive/AMIR Lab/Research Assis...,/content/drive/MyDrive/AMIR Lab/Research Assis...,regenerated,True,single_bar:macro_f1
5,phase03g_clean_input_ablation_macro_f1_compari...,/content/drive/MyDrive/AMIR Lab/Research Assis...,/content/drive/MyDrive/AMIR Lab/Research Assis...,regenerated,True,single_bar:test_macro_f1_mean
6,phase03g_clean_input_ablation_metric_panel.png,/content/drive/MyDrive/AMIR Lab/Research Assis...,/content/drive/MyDrive/AMIR Lab/Research Assis...,regenerated,True,"metrics:test_accuracy_mean,test_macro_f1_mean,..."
7,phase03g_qadp_macro_f1_gain_over_input_ablatio...,/content/drive/MyDrive/AMIR Lab/Research Assis...,/content/drive/MyDrive/AMIR Lab/Research Assis...,regenerated,True,single_bar:qadp_minus_reference
8,phase03g_question_only_strongest_slices.png,/content/drive/MyDrive/AMIR Lab/Research Assis...,/content/drive/MyDrive/AMIR Lab/Research Assis...,regenerated,True,single_bar:macro_f1_mean
9,phase5b_clean_metadata_four_seeds_answer_backb...,/content/drive/MyDrive/AMIR Lab/Research Assis...,/content/drive/MyDrive/AMIR Lab/Research Assis...,regenerated,True,"metrics:test_accuracy_mean,test_macro_f1_mean,..."


Strict figure coverage validation passed.
FIGURE_COVERAGE_PASSED: True


## Write Manifest And Validation Report


In [ ]:
manifest_dir = OUTPUT_ROOT / "manifests"
manifest_dir.mkdir(parents=True, exist_ok=True)

manifest_df = pd.DataFrame(manifest_rows)
manifest_path = manifest_dir / "paper_ready_artifact_manifest.csv"
manifest_df.to_csv(manifest_path, index=False)

source_validation_path = manifest_dir / "source_validation_report.csv"
pd.DataFrame(validation_rows).to_csv(source_validation_path, index=False)

table_schema_path = manifest_dir / "table_schema_validation_report.csv"
table_schema_validation_df.to_csv(table_schema_path, index=False)

figure_status_path = manifest_dir / "figure_regeneration_status.csv"
figure_regeneration_df.to_csv(figure_status_path, index=False)

old_display_hits = []
allowed_provenance_hits = []

def record_old_label_hit(path: Path, location: str, key: str, value: str) -> None:
    for pattern in OLD_LABEL_PATTERNS:
        if pattern in value:
            hit = {
                "destination_path": str(path),
                "location": location,
                "field_or_key": key,
                "pattern": pattern,
                "value_preview": value[:240],
            }
            if is_provenance_key(key) or any(token in value.lower() for token in ["/content/drive", "mydrive"]):
                allowed_provenance_hits.append(hit)
            else:
                old_display_hits.append(hit)


def scan_json_values(path: Path, obj: Any, key: str = "", location: str = "$") -> None:
    if isinstance(obj, dict):
        for k, v in obj.items():
            scan_json_values(path, v, str(k), f"{location}.{k}")
    elif isinstance(obj, list):
        for index, item in enumerate(obj):
            scan_json_values(path, item, key, f"{location}[{index}]")
    elif isinstance(obj, str):
        record_old_label_hit(path, location, key, obj)


for row in manifest_rows:
    dst = Path(row["destination_path"])
    suffix = dst.suffix.lower()
    if suffix not in TEXT_SUFFIXES:
        continue
    if suffix == ".csv":
        df = pd.read_csv(dst, dtype=str, keep_default_na=False)
        for col in df.columns:
            for row_index, value in df[col].items():
                if isinstance(value, str):
                    record_old_label_hit(dst, f"row={row_index}", col, value)
    elif suffix == ".json":
        scan_json_values(dst, json.loads(dst.read_text(encoding="utf-8")), location="$")
    elif suffix == ".jsonl":
        for line_number, line in enumerate(dst.read_text(encoding="utf-8").splitlines(), start=1):
            if line.strip():
                scan_json_values(dst, json.loads(line), location=f"line={line_number}")
    else:
        for line_number, line in enumerate(dst.read_text(encoding="utf-8", errors="replace").splitlines(), start=1):
            key = "provenance_line" if any(token in line.lower() for token in ["source", "path", "provenance", "/content/drive", "mydrive", "artifact"]) else "display_text"
            record_old_label_hit(dst, f"line={line_number}", key, line)

validation_report = {
    "run_id": RUN_ID,
    "framework_statement": FRAMEWORK_STATEMENT,
    "project_root": str(PROJECT_ROOT),
    "release_base_dir": str(RELEASE_BASE_DIR),
    "output_root": str(OUTPUT_ROOT),
    "architecture_figure_source_dir": str(ARCHITECTURE_FIGURE_SOURCE_DIR) if ARCHITECTURE_FIGURE_SOURCE_DIR else None,
    "architecture_context": architecture_context_rows,
    "embedded_local_context_policy": "Paper Writing is local-only; naming context was embedded into this notebook at build time.",
    "source_validation": validation_rows,
    "num_planned_artifacts": len(copy_plan),
    "num_copied_artifacts": len(manifest_rows),
    "num_text_artifacts_rewritten": int((manifest_df["rewrite_count"] > 0).sum()) if not manifest_df.empty else 0,
    "total_rewrite_count": int(manifest_df["rewrite_count"].sum()) if not manifest_df.empty else 0,
    "num_generated_figures": len(generated_figures),
    "generated_figures": [str(path) for path in generated_figures],
    "num_paper_use_allowed_figures": int(figure_regeneration_df["paper_use_allowed"].sum()) if not figure_regeneration_df.empty else 0,
    "num_blocked_result_figures": int(figure_regeneration_df["status"].isin(["source_copy_only_blocked", "missing_source_table_blocked"]).sum()) if not figure_regeneration_df.empty else 0,
    "figure_coverage_passed": bool(FIGURE_COVERAGE_PASSED),
    "old_label_blocking_hits": old_display_hits,
    "old_label_allowed_provenance_hits": allowed_provenance_hits[:200],
    "old_label_policy": "Old names are allowed only in provenance/source-path fields; review remaining hits manually.",
    "preservation_policy": "No source artifacts were moved, deleted, or edited. This notebook only copies into OUTPUT_ROOT.",
    "metrics_policy": "Metrics are not recalculated; regenerated figures use copied validated result tables.",
}

report_path = manifest_dir / "paper_ready_release_report.json"
report_path.write_text(json.dumps(validation_report, indent=2, ensure_ascii=False), encoding="utf-8")

readme_path = manifest_dir / "README_PAPER_READY_RELEASE.md"
readme_path.write_text(
    "# AUGR-VQA Paper-Ready Release\n\n"
    f"Framework: {FRAMEWORK_STATEMENT}\n\n"
    "This folder is the paper-facing ground-truth package generated by Phase 07.\n\n"
    "- Original phase folders were not edited.\n"
    "- Metrics were not recalculated.\n"
    "- Text/table/report copies were harmonized to paper naming.\n"
    "- Paper-use result figures are in `figures/regenerated_paper_ready/`.\n"
    "- Architecture figures are in `figures/architecture/` only if `ARCHITECTURE_FIGURE_SOURCE_DIR` was set before copying.\n"
    "- Old notebook PNGs are in `figures/source_copies/` for traceability only.\n"
    "- Phase 05D/support reference figures are in `figures/reference_copies/` and are not direct paper-use figures.\n"
    "- `Paper Writing/` was not read from Google Drive; paper naming context was embedded in this notebook at build time.\n"
    "- Use `manifests/figure_regeneration_status.csv` to verify paper-use status.\n",
    encoding="utf-8",
)

if old_display_hits:
    print("ERROR: old proposed-model labels remain outside allowed provenance fields.")
    display(pd.DataFrame(old_display_hits).head(50))
    raise RuntimeError("Blocking old labels remain in copied text artifacts.")
elif not FIGURE_COVERAGE_PASSED:
    print("ERROR: strict result figure coverage failed. See manifests/figure_regeneration_status.csv.")
    display(figure_regeneration_df[figure_regeneration_df["status"].isin(["source_copy_only_blocked", "missing_source_table_blocked"])].head(50))
    raise RuntimeError("Strict figure regeneration failed. Diagnostics were written to the release manifests.")
else:
    print("No blocking old proposed-model display labels found in copied text artifacts.")

print(f"Manifest written: {manifest_path}")
print(f"Source validation written: {source_validation_path}")
print(f"Table schema validation written: {table_schema_path}")
print(f"Figure regeneration status written: {figure_status_path}")
print(f"Release report written: {report_path}")
print(f"Release README written: {readme_path}")


No blocking old proposed-model display labels found in copied text artifacts.
Manifest written: /content/drive/MyDrive/AMIR Lab/Research Assistant Intern/research_01_brain_tumor_VQA_medical_imaging/phase_7_paper_ready_augr_vqa_release/btumqa_225k_clean_metadata_four_seeds/results/manifests/paper_ready_artifact_manifest.csv
Source validation written: /content/drive/MyDrive/AMIR Lab/Research Assistant Intern/research_01_brain_tumor_VQA_medical_imaging/phase_7_paper_ready_augr_vqa_release/btumqa_225k_clean_metadata_four_seeds/results/manifests/source_validation_report.csv
Table schema validation written: /content/drive/MyDrive/AMIR Lab/Research Assistant Intern/research_01_brain_tumor_VQA_medical_imaging/phase_7_paper_ready_augr_vqa_release/btumqa_225k_clean_metadata_four_seeds/results/manifests/table_schema_validation_report.csv
Figure regeneration status written: /content/drive/MyDrive/AMIR Lab/Research Assistant Intern/research_01_brain_tumor_VQA_medical_imaging/phase_7_paper_ready_aug

## Final Release Summary


In [ ]:
print("Phase 07 paper-ready release complete.")
print(FRAMEWORK_STATEMENT)
print(f"Output folder: {OUTPUT_ROOT}")
print(f"Copied artifacts: {len(manifest_rows)}")
print(f"Generated figures: {len(generated_figures)}")
print("Original notebooks and result folders were not edited.")


Phase 07 paper-ready release complete.
AUGR-VQA = QAdp-DG-PRUGTM + Q-CUR
Output folder: /content/drive/MyDrive/AMIR Lab/Research Assistant Intern/research_01_brain_tumor_VQA_medical_imaging/phase_7_paper_ready_augr_vqa_release/btumqa_225k_clean_metadata_four_seeds/results
Copied artifacts: 698
Generated figures: 43
Original notebooks and result folders were not edited.
